# Retrieval

We have 102 chunks stored in ChromaDB with embeddings from 1_load_chunk.

This notebook builds the retrieval layer:
- Take a plain text query
- Embed it using the same model
- Find the top-k most similar chunks using similarity search
- Inspect the results

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [2]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


## Load the vector store

In [3]:
use_OpenAI = True

if use_OpenAI:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectorstore = Chroma(
        persist_directory="../chroma_DB/",
        embedding_function=embeddings
    )

    print(f"Chunks in vectorstore: {vectorstore._collection.count()}")

C:\Users\RAZER\AppData\Local\Temp\ipykernel_21140\3895086237.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chunks in vectorstore: 102


## Basic similarity search

`similarity_search()` takes a plain string, embeds it automatically, and returns the top-k most similar chunks.

`k` controls how many chunks come back — this is the **top-k** parameter. Higher k = more context, but also more noise. `k=3` is a reasonable default to start.

In [5]:
query = "What does the moth look like?"
k = 3

if use_OpenAI:
    results = vectorstore.similarity_search(query, k=k)

    print(f"Query: '{query}'")
    print(f"Top {k} results:\n")

    for i, doc in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(doc.page_content)
        print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'What does the moth look like?'
Top 3 results:

--- Result 1 ---
Source: ..\data\1_source\wikipedia_article_1.txt
The moth is a fairly large, heavy-bodied species with a wingspan of 55–68 mm (2.2–2.7 in). The forewings are grey with a large prominent buff patch at the apex. As the thoracic hair is also buff, the moth resembles a broken twig when at rest. The hindwings are creamy white. Seitz - Head, collar and centre of thorax brownish yellow, patagia greyish white with a black-brown double basal edge, on the transverse crest 2 black-brown transverse lines, hind margin greyish white. Abdomen yellowish grey

--- Result 2 ---
Source: ..\data\1_source\wikipedia_article_1.txt
The moth flies at night in June and July[a] and sometimes comes to light, although it is not generally strongly attracted.

The young larvae are gregarious, becoming solitary later. The older larva is very striking, black with white and yellow lines. It feeds on many trees and shrubs (see list below). The speci

## Try a different query

Swap in a query that should pull different chunks — this is a good way to build intuition for how semantic search behaves vs. keyword search.

In [6]:
query2 = "Where is this species found geographically?"
k = 3

if use_OpenAI:
    results2 = vectorstore.similarity_search(query2, k=k)

    print(f"Query: '{query2}'")
    print(f"Top {k} results:\n")

    for i, doc in enumerate(results2):
        print(f"--- Result {i+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(doc.page_content)
        print()

Query: 'Where is this species found geographically?'
Top 3 results:

--- Result 1 ---
Source: ..\data\1_source\wikipedia_article_1.txt
of the Arctic Region and Greece; also in North-East Africa, Asia Minor, Siberia to East Asia. In Central Europe abundant everywhere in May and June, a second brood in July and August appears regularly only in the South. — In Norway and Southern Sweden, also in England occurs a dark form, tenebrata Strand, [subspecies P. b. tenebrata Strand, 1903] in which the white colouring of the forewing is more or less strongly reduced, particularly in the median area, while the hindwing is paler or darker

--- Result 2 ---
Source: ..\data\1_source\wikipedia_article_1.txt
area, while the hindwing is paler or darker grey. In ab. demaculata Strand (47 d) [aberration] the pale discal spot of the forewing moreover is absent. — bucephalina Stgr.,[ now species Phalera bucephalina (Staudinger & Rebel, 1901)] which represents the species in Western Morocco, is also characte

## Wrap it in a reusable function

Once this is solid, we would move it to `src/helpers.py`
Then edit this notebook to import this, and apply it

In [ ]:
def retrieve(query, vectorstore, k=3):
    """
    Given a query string, return the top-k most relevant chunks from the vectorstore.
    Returns a list of LangChain Document objects.
    """
    results = vectorstore.similarity_search(query, k=k)
    return results


def print_results(query, results):
    print(f"Query: '{query}'\n")
    for i, doc in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(doc.page_content)
        print()




In [8]:
# Test the functions
if use_OpenAI:
    test_query = "How does the moth camouflage itself?"
    test_results = retrieve(test_query, vectorstore, k=3)
    print_results(test_query, test_results)

Query: 'How does the moth camouflage itself?'

--- Result 1 ---
Source: ..\data\1_source\wikipedia_article_1.txt
The moth is a fairly large, heavy-bodied species with a wingspan of 55–68 mm (2.2–2.7 in). The forewings are grey with a large prominent buff patch at the apex. As the thoracic hair is also buff, the moth resembles a broken twig when at rest. The hindwings are creamy white. Seitz - Head, collar and centre of thorax brownish yellow, patagia greyish white with a black-brown double basal edge, on the transverse crest 2 black-brown transverse lines, hind margin greyish white. Abdomen yellowish grey

--- Result 2 ---
Source: ..\data\1_source\wikipedia_article_1.txt
The moth flies at night in June and July[a] and sometimes comes to light, although it is not generally strongly attracted.

The young larvae are gregarious, becoming solitary later. The older larva is very striking, black with white and yellow lines. It feeds on many trees and shrubs (see list below). The species overw

## Summary

- `similarity_search(query, k=k)` is all it takes to retrieve relevant chunks
- The query is embedded on the fly using the same model as the stored chunks
- `k` is a key parameter — experiment with 3, 5, and 10 to see how result quality changes
- The `retrieve()` function above is ready to be moved to `src/helpers.py`

**Next (notebook 4):** pass these retrieved chunks as context to an LLM and generate an answer — the full RAG loop.